In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import numpy as np
from scipy.stats import binom_test
from scipy.stats import binom
import argparse
import json
import configparser

np.seterr(invalid='ignore')

{'divide': 'warn', 'over': 'warn', 'under': 'ignore', 'invalid': 'warn'}

In [2]:
DATABASE = "../../drive_data/v33_koondkorpus_sentences_verb_pattern_obl_20241002-130310_gpt_labelled.db"

OBL_TABLE = f"gpt_labelled_clean"


In [3]:

def examples_table(database, obl_table):
    """Gets all verb+comp+case plus tags and example sentences from spatial_obl (for hoverplot).
    Calculates
    - how many rows belong to each tag (ELT, A, S, or empty),
    - percentages per tag,
    - and up to 3 random example sentences for each tag+verbcase combination.
    Tables (not permanent):
    base: spatial obl join transaction_head. 0 or 1 for each tag if it is in tags column.
    aggregated: statistics. Groups by verbcase, what is the total count, count for each tag and count for no tag.
    long_counts: transforms wide counts into long format. Each row is verbcase,total,tag, count (each row has count for one tag).
    percentages: percentages for each tag in verbcase (based on long_counts).
    tagged_rows:  creates a normalized tag-level example table (from base).
    ranked_examples: randomly ranks/numbers example rows within each verbcase+tag.
    top_examples: takes 3 random examples per subgroup (verbcase+tag).
    examples_json: packages example rows into JSON arrays.
    final result: joins percentages and examples_json. 
    output format:
    verb	compound	case	tag	count	total	    pct	examples
    v1	    comp1	    ALL     A	    50	    100	    50.0	[...]
    v1	    comp1	    ALL     ELT	20	    100	    20.0	[...]
    
    """

    conn = sqlite3.connect(database)

    query = f"""
    WITH base AS (
        SELECT
            so.verb,
            so.verb_compound,
            so.morph_case,
            so.form,
            so.head_loc,
            so.sentence,
            so.sentence_id,
            th.phrase,
            th.form as verb_form,
            so.gpt_tags,

            CASE WHEN so.gpt_tags LIKE '%|ELT|%' THEN 1 ELSE 0 END AS is_elt,
            CASE WHEN so.gpt_tags LIKE '%|A|%'   THEN 1 ELSE 0 END AS is_a,
            CASE WHEN so.gpt_tags LIKE '%|S|%'   THEN 1 ELSE 0 END AS is_s,
            CASE WHEN so.gpt_tags = '' OR so.gpt_tags IS NULL THEN 1  ELSE 0 END AS is_empty

        FROM {obl_table} so

        LEFT JOIN transaction_head th
          ON so.sentence_id = th.sentence_id
         AND so.verb = th.verb
         AND so.verb_compound = th.verb_compound
        AND so.head_id = th.id
    ),

    aggregated AS (
        SELECT
            verb,
            verb_compound,
            morph_case,

            COUNT(*) AS total,

            SUM(is_elt)   AS elt_count,
            SUM(is_a)     AS a_count,
            SUM(is_s)     AS s_count,
            SUM(is_empty) AS empty_count

        FROM base

        GROUP BY
            verb,
            verb_compound,
            morph_case
    ),

    long_counts AS (

        SELECT
            verb,
            verb_compound,
            morph_case,
            total,
            'ELT' AS tag,
            elt_count AS count
        FROM aggregated

        UNION ALL

        SELECT
            verb,
            verb_compound,
            morph_case,
            total,
            'A',
            a_count
        FROM aggregated

        UNION ALL

        SELECT
            verb,
            verb_compound,
            morph_case,
            total,
            'S',
            s_count
        FROM aggregated

        UNION ALL

        SELECT
            verb,
            verb_compound,
            morph_case,
            total,
            '' AS tag,
            empty_count
        FROM aggregated
    ),

    percentages AS (
        SELECT
            *,
            ROUND(100.0 * count / total, 2) AS pct
        FROM long_counts
    ),

    tagged_rows AS (

        SELECT
            verb,
            verb_compound,
            morph_case,
            sentence,
            sentence_id,
            form,
            phrase,
            verb_form,
            head_loc,
            'ELT' AS tag

        FROM base
        WHERE is_elt = 1

        UNION ALL

        SELECT
            verb,
            verb_compound,
            morph_case,
            sentence,
            sentence_id,
            form,
            phrase,
            verb_form,
            head_loc,
            'A'

        FROM base
        WHERE is_a = 1

        UNION ALL

        SELECT
            verb,
            verb_compound,
            morph_case,
            sentence,
            sentence_id,
            form,
            phrase,
            verb_form,
            head_loc,
            'S'

        FROM base
        WHERE is_s = 1

        UNION ALL

        SELECT
            verb,
            verb_compound,
            morph_case,
            sentence,
            sentence_id,
            form,
            phrase,
            verb_form,
            head_loc,
            '' AS tag

        FROM base
        WHERE is_empty = 1
    ),

    ranked_examples AS (
        SELECT
            verb,
            verb_compound,
            morph_case,
            tag,
            sentence,
            sentence_id,
            form,
            phrase,
            verb_form,
            head_loc,

            ROW_NUMBER() OVER (
                PARTITION BY
                    verb,
                    verb_compound,
                    morph_case,
                    tag
                ORDER BY RANDOM()
            ) AS rn

        FROM tagged_rows
    ),

    top_examples AS (
        SELECT *
        FROM ranked_examples
        WHERE rn <= 3
    ),

    examples_json AS (
        SELECT
            verb,
            verb_compound,
            morph_case,
            tag,
            -- information that is needed for highlighting/marking for plotting
            json_group_array(
                json_object(
                    'sentence', sentence,
                    'sentence_id', sentence_id,
                    'form', form,
                    'phrase', phrase,
                    'verb_form', verb_form,
                    'verb_compound', verb_compound,
                    'head_loc', head_loc
                )
            ) AS examples

        FROM top_examples

        GROUP BY
            verb,
            verb_compound,
            morph_case,
            tag
    )

    SELECT
        p.verb,
        p.verb_compound,
        p.morph_case,
        p.tag,
        p.count,
        p.total,
        p.pct,
        e.examples

    FROM percentages p

    LEFT JOIN examples_json e
      ON p.verb = e.verb
     AND p.verb_compound = e.verb_compound
     AND p.morph_case = e.morph_case
     AND p.tag = e.tag

    ORDER BY
        p.verb,
        p.verb_compound,
        p.morph_case,
        p.pct DESC;
    """

    df2 = pd.read_sql(query, conn)

    conn.close()

    return df2


In [4]:
examples_df = examples_table(DATABASE, OBL_TABLE)

In [5]:
examples_df

,verb,verb_compound,morph_case,tag,count,total,pct,examples
0,abistama,,ad,ELT,228,301,75.75,"[{""sentence"":""Ettevõtluskeskuste konsultandid ..."
1,abistama,,ad,,64,301,21.26,"[{""sentence"":""Isa puudumisel , abistas Tammert..."
2,abistama,,ad,A,5,301,1.66,"[{""sentence"":""Austraalias lugesin lehest , kui..."
3,abistama,,ad,S,4,301,1.33,"[{""sentence"":""Ma usun , et ka meie parempoolse..."
4,abistama,,in,ELT,116,139,83.45,"[{""sentence"":""Komisjoni abistab arenguküsimust..."
...,...,...,...,...,...,...,...,...
8923,šokeerima,,ad,S,0,58,0.00,None
8924,šokeerima,,in,ELT,36,38,94.74,"[{""sentence"":""Paide noor kunstnik Gert Hatsuko..."
8925,šokeerima,,in,A,1,38,2.63,"[{""sentence"":""Galginaitis , keda hinnatakse tu..."
8926,šokeerima,,in,S,1,38,2.63,"[{""sentence"":""Kui \"" Lilja 4-ever \"" šokeeris ..."


In [6]:
conn = sqlite3.connect(DATABASE)
cursor = conn.cursor()

examples_df.to_sql(f"gpt_labelled_all_plotting_examples", conn, if_exists="replace", index=False)

conn.close()